# Wrapping the QA Agent into an A2A Server

In this lesson, you will take the `PolicyAgent` created in Lesson 3 and wrap it into an Agent2Agent (A2A) server. This makes the agent discoverable and callable by other agents using the A2A protocol. You will define the agent's identity using an `AgentCard`, describe its capabilities using `AgentSkill`, and set up the request handling logic.

## 4.1. Define the A2A Server

You will write the server code to a file named `a2a_policy_agent.py`. This script performs the following steps:

1.  **Imports**: Loads necessary modules including the A2A SDK (`a2a.server`).

2.  **Executor**: Defines `PolicyAgentExecutor`, which bridges the A2A request context to your `PolicyAgent` class.

3.  **Metadata**: Configures the `AgentSkill` (defining *what* it can do) and the `AgentCard` (defining *who* it is, including its URL).

4.  **Application**: Creates and runs an `A2AStarletteApplication` using `uvicorn`.

In [ ]:
%%writefile a2a_server_policy_agent.py

# ─────────────────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────────────────

# load_dotenv reads key=value pairs from a .env file and puts them into
# os.environ so your code can access API keys / config without hardcoding them.
from dotenv import load_dotenv
import os

# uvicorn is an ASGI web server — it is the process that actually listens on
# a port and forwards incoming HTTP requests to our application.
import uvicorn

# AgentExecutor  – abstract base class you must subclass.  The A2A framework
#                  calls its `execute()` method whenever a new task arrives.
# RequestContext – wraps the incoming A2A request; provides helpers like
#                  get_user_input() so you don't parse raw JSON yourself.
from a2a.server.agent_execution import AgentExecutor, RequestContext

# A2AStarletteApplication – turns your AgentCard + RequestHandler into a
#   fully-spec-compliant A2A HTTP application (built on the Starlette ASGI
#   framework).  It exposes:
#     GET  /.well-known/agent.json  →  AgentCard (agent discovery)
#     POST /                        →  task execution endpoint
from a2a.server.apps import A2AStarletteApplication

# EventQueue – an async queue the executor writes events (messages, artifacts,
#   status updates) into.  The framework drains it and sends the events back
#   to the calling client, supporting both streaming and non-streaming modes.
from a2a.server.events import EventQueue

# DefaultRequestHandler – implements the full A2A task lifecycle on your behalf
#   (receiving the request, creating a Task object, calling the executor,
#   collecting events, and returning the final response).
from a2a.server.request_handlers import DefaultRequestHandler

# InMemoryTaskStore – stores Task objects in a Python dict (in RAM).
#   Fine for development / single-process servers.  For production you would
#   swap this for a persistent store (Redis, PostgreSQL, etc.).
from a2a.server.tasks import InMemoryTaskStore

# AgentCapabilities – flags that advertise what the agent supports
#                     (e.g. streaming=True/False, pushNotifications, etc.)
# AgentCard         – the agent's "identity card" returned at
#                     GET /.well-known/agent.json; other agents discover this
#                     URL to learn the agent's name, skills, and endpoint.
# AgentSkill        – describes ONE capability the agent exposes (id, name,
#                     description, example prompts).  An AgentCard can list
#                     multiple skills.
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentSkill
)

# new_agent_text_message – convenience factory that builds an A2A-compliant
#   Message object containing a plain-text Part, saving you from constructing
#   the nested JSON structure by hand.
from a2a.utils import new_agent_text_message

# Your own domain agent (defined in agents.py in the same directory).
# PolicyAgent contains the actual LLM + RAG logic that answers insurance
# policy questions.  The A2A layer is just a network wrapper around it.
from agents import PolicyAgent


# ─────────────────────────────────────────────────────────────────────────────
# EXECUTOR  (the bridge between A2A protocol ↔ your agent logic)
# ─────────────────────────────────────────────────────────────────────────────

class PolicyAgentExecutor(AgentExecutor):
    """
    Subclass of AgentExecutor — ONE executor per logical agent (or skill group).
    Its job: pull the user's message out of the A2A context, call your domain
    agent, then push the result back through the EventQueue.

    Best practice: 1 AgentExecutor : 1 AgentCard.
    If you need multiple distinct skills, you can handle them inside a single
    executor (by inspecting context.skill_id) OR deploy separate agents with
    their own AgentCard + executor.
    """

    def __init__(self) -> None:
        # Instantiate your real agent once at startup (not per-request) so
        # expensive initialisation (model loading, index loading) happens only
        # once.  This instance is reused for every incoming task.
        self.agent = PolicyAgent()

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        """
        Called by DefaultRequestHandler for every new task.

        Parameters
        ----------
        context    : Provides the parsed A2A task, including the user's message.
        event_queue: Async queue to publish response events back to the caller.
        """
        # Extract the plain-text user message from the A2A Task object.
        # context.get_user_input() navigates the nested A2A message/part
        # structure so you don't have to.
        prompt = context.get_user_input()

        # Delegate to your domain agent — this is where the LLM / RAG call
        # actually happens.  The A2A layer doesn't care how this works.
        response = self.agent.answer_query(prompt)

        # Wrap the plain-text answer in an A2A-compliant Message object.
        # new_agent_text_message creates: Message → Part → TextPart(text=response)
        message = new_agent_text_message(response)

        # Publish the message event.  The framework reads from this queue and
        # sends the response back to the client (HTTP body for non-streaming,
        # SSE chunks for streaming).
        await event_queue.enqueue_event(message)

    def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        """
        Called when the client sends a cancel request for an in-flight task.
        Implement task interruption logic here (e.g., cancel an async LLM call).
        Left as `pass` here because PolicyAgent's synchronous calls cannot be
        cancelled mid-flight.
        """
        pass


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

def main() -> None:
    print(f"Running A2A Health Insurance Policy Agent")

    # Read API keys and config from the .env file into os.environ.
    # The underscore discards the return value (True/False) — we don't need it.
    _ = load_dotenv()

    # Read the port from the environment, default to 9999.
    # int() is needed because os.environ returns strings.
    PORT = int(os.environ.get("POLICY_AGENT_PORT", 9999))

    # Read the host (IP / hostname) the server should bind to.
    HOST = os.environ.get("AGENT_HOST", "localhost")

    # ── SKILL ──────────────────────────────────────────────────────────────
    # AgentSkill describes ONE capability this agent offers.
    # Clients (and orchestrators) use this to decide which agent to call.
    #
    # id          – machine-readable identifier; must be unique within this card.
    # name        – human-readable label shown in UIs.
    # description – natural-language summary used by LLM orchestrators to
    #               decide whether this skill matches a user's intent.
    # tags        – keywords for filtering / search.
    # examples    – sample prompts that illustrate the skill's scope; also
    #               used by orchestrators for few-shot routing decisions.
    #
    # BEST PRACTICE: 1 AgentCard can list multiple skills, but each skill
    # should be distinct enough that an orchestrator can reliably choose
    # among them.  If skills are very different in domain, consider separate
    # agents so each has a focused, easily-discoverable AgentCard.
    skill = AgentSkill(
        id="insurance_coverage",
        name="Insurance Coverage",
        description="Provides information about insurance coverage options and details.",
        tags=["insurance", "coverage"],
        examples=["What does my policy cover?", "Are mental health services included?"]
    )

    # ── AGENT CARD ─────────────────────────────────────────────────────────
    # AgentCard is the agent's public identity, published at:
    #   GET http://<HOST>:<PORT>/.well-known/agent.json
    # Any client (human or agent) fetches this URL to discover:
    #   • what the agent can do (skills)
    #   • what input/output data formats it accepts (modes)
    #   • whether it supports streaming
    #   • the endpoint URL to POST tasks to
    #
    # default_input_modes  – data formats the agent accepts:
    #     "text"  → plain UTF-8 string
    #     "file"  → binary file (image, PDF, audio …)
    #     "data"  → structured JSON
    #     Agents can list multiple modes; the client picks the best one.
    # default_output_modes – same concept for responses.
    # capabilities         – feature flags:
    #     streaming=True  → server supports SSE (server-sent events) for
    #                        incremental token-by-token output.
    #     streaming=False → full response returned in one HTTP reply.
    # skills               – list of AgentSkill objects (one or many).
    agent_card = AgentCard(
        name="InsurancePolicyCoverageAgent",
        description="Provides information about insurance policy coverage options and details.",
        # The URL where clients POST new tasks.  Must be publicly reachable
        # by any agent that wants to call us.
        url=f"http://{HOST}:{PORT}/",
        version="1.0.0",
        default_input_modes=["text"],   # we accept plain text prompts
        default_output_modes=["text"],  # we return plain text answers
        capabilities=AgentCapabilities(streaming=False),  # no SSE streaming
        skills=[skill]  # list all skills this agent can perform
    )

    # ── REQUEST HANDLER ────────────────────────────────────────────────────
    # DefaultRequestHandler implements the full A2A task lifecycle:
    #   1. Validate the incoming JSON-RPC request.
    #   2. Create (or resume) a Task object in the task_store.
    #   3. Call executor.execute() to run your agent logic.
    #   4. Collect events from the EventQueue.
    #   5. Write the final Task state back to the store.
    #   6. Return the Task (or stream events) to the client.
    #
    # agent_executor – your PolicyAgentExecutor defined above.
    # task_store     – where Task objects are persisted between steps.
    #                  InMemoryTaskStore = dict in RAM (no persistence across
    #                  restarts; fine for stateless Q&A agents like this one).
    request_handler = DefaultRequestHandler(
        agent_executor=PolicyAgentExecutor(),
        task_store=InMemoryTaskStore()
    )

    # ── A2A APPLICATION ────────────────────────────────────────────────────
    # A2AStarletteApplication wires everything together into a Starlette ASGI
    # application with two routes:
    #   GET  /.well-known/agent.json  →  serves the AgentCard JSON
    #   POST /                        →  routes JSON-RPC calls to request_handler
    #
    # agent_card   – served at the discovery endpoint.
    # http_handler – handles incoming task requests.
    server = A2AStarletteApplication(
        agent_card=agent_card,
        http_handler=request_handler
    )

    # server.build() returns the raw ASGI callable that uvicorn needs.
    # uvicorn starts the event loop, binds to HOST:PORT, and serves requests.
    uvicorn.run(server.build(), host=HOST, port=PORT)


if __name__ == '__main__':
    main()

## 4.2. Run the Policy A2A Server

Now to activate your configured A2A agent, you would need to run your agent server. You can run the agent server using `uv`:

- Open Terminal 1 by running the cell below

- Type `uv run a2a_policy_agent.py` to run the server and activate your A2A agent.

- When running the agent for the first time, you will see that the virtual environment will first be created automatically and then the agent will run. In the video, the virtual environment was already created.

**Note:** If you go to the next lesson, this agent will keep on running unless you manually stop it (`CTRL+C`) or the lab environment resets (on this learning platform, the lab environment resets after 30 minutes of inactivity or automatically every 2 hours).

If you stop the agent or the lab environment restarts, you don't need to come back to this lesson to launch the agent again. You will be provided with the needed files and terminal in each lesson, so you can always continue from where you left off.

In [ ]:
import os

from IPython.display import IFrame

url = os.environ.get("DLAI_LOCAL_URL").format(port=8888)
IFrame(f"{url}terminals/1", width=550, height=600)